MetroStack Audit Crew 
What this does: Runs the MetroStack 3-agent audit pipeline 
Before running:
Upload your MetroStack folder 
Upload your GGUF model files 
Attach both datasets to this notebook 
Run all cells top to bottom — or just Run All


In [1]:
# FILESYTEM INITIALIZATION MODE 


import os
import sys
from pathlib import Path

# 1. Inspect the current sandboxed environment
print("=== Sandboxed Filesystem Inspection ===")
current_working_dir = Path(os.getcwd())
print(f"Current Virtual Working Directory: {current_working_dir}")

# List everything currently available in the virtual space
print("\nAvailable items in virtual root:")
try:
    for item in os.listdir('.'):
        is_dir = "[DIR]" if os.path.path.isdir(item) else "[FILE]"
        print(f"  {is_dir} {item}")
except Exception as e:
    print(f"  Error scanning directory: {e}")

print("\n=== Dynamic Path Configuration ===")

# 2. Safely configure paths. 
# If running in a true local server (like JupyterLab), it uses the real parent.
# If running in the browser html runner, it sets up a virtual directory path.
if os.path.exists('/kaggle/input'):
    # Cloud environment fallback
    DATA_SRC = Path('/kaggle/input')
    print("Environment Detected: Kaggle Cloud Server")
elif current_working_dir.name == "src" or (current_working_dir / '..').resolve().exists():
    # True local development fallback
    DATA_SRC = (current_working_dir / '..').resolve()
    print("Environment Detected: Local Development (Parent Directory Accessible)")
else:
    # Browser Sandbox Fallback
    # Automatically creates a mock input tree inside the browser's virtual memory
    DATA_SRC = Path('/virtual/input')
    os.makedirs(DATA_SRC, exist_ok=True)
    print("Environment Detected: Browser Wasm Sandbox")
    print(f"-> Virtual input directory initialized at: {DATA_SRC}")

# 3. Assign your global project path variables
# Change 'retrostack' to whatever project folder name your notebook needs
PROJECT_ROOT = DATA_SRC / 'Metrostack'
print(f"\nTarget Project Variable Set:")
print(f"  PROJECT_ROOT = {PROJECT_ROOT}")

# Inject into global variables so all cells below can safely use them
# Example: with open(PROJECT_ROOT / 'config.json') as f:

=== Sandboxed Filesystem Inspection ===
Current Virtual Working Directory: /home/pyodide

Available items in virtual root:

=== Dynamic Path Configuration ===
Environment Detected: Local Development (Parent Directory Accessible)

Target Project Variable Set:
  PROJECT_ROOT = /home/retrostack


Cell 2 — Build llama.cpp with CUDA support


In [4]:
import os
from pathlib import Path

# 1. Point to the current working directory where you uploaded your files
# In MetroStack, this is the root of the virtual filesystem
WORKSPACE_DIR = Path('.')

# 2. Look for the pre-built binary and the model you uploaded via the sidebar
SERVER_BIN = WORKSPACE_DIR / 'llama-server'
MODEL_FILE = WORKSPACE_DIR / 'model.gguf' # Replace with your actual model name

print("[INFO] MetroStack: Verifying pre-compiled environment...")

# 3. Check if the files are present (these must be uploaded via the UI first)
if not SERVER_BIN.exists():
    print(f"[ERROR] Binary not found at {SERVER_BIN}. Please upload the pre-compiled llama-server binary.")
else:
    os.environ['LLAMA_SERVER_BIN'] = str(SERVER_BIN)
    print(f"[OK] Binary path set: {os.environ['LLAMA_SERVER_BIN']}")

if not MODEL_FILE.exists():
    print(f"[WARN] Model not found at {MODEL_FILE}. Please upload your .gguf model.")
else:
    print(f"[OK] Model found: {MODEL_FILE.name}")

print("[INFO] Ready to run server.")

[INFO] MetroStack: Verifying pre-compiled environment...
[ERROR] Binary not found at llama-server. Please upload the pre-compiled llama-server binary.
[WARN] Model not found at model.gguf. Please upload your .gguf model.
[INFO] Ready to run server.


Cell 3 — Start Junior and Senior LLM servers


In [ ]:
import subprocess, time, os, urllib.request
from pathlib import Path

MODELS_DIR   = Path('/kaggle/input/metrostack-models')
SERVER_BIN   = os.environ.get('LLAMA_SERVER_BIN')
LOG_DIR      = Path('/kaggle/working/logs')
LOG_DIR.mkdir(exist_ok=True)

# ── Model file mapping ────────────────────────────────────────────
# These filenames must match what you uploaded to the models dataset
JUNIOR_MODEL = MODELS_DIR / 'qwen2.5-coder-1.5b-instruct-q4_k_m.gguf'
SENIOR_MODEL = MODELS_DIR / 'deepseek-coder-1.3b-instruct.Q4_K_M.gguf'

for m in [JUNIOR_MODEL, SENIOR_MODEL]:
    if not m.exists():
        raise FileNotFoundError(f'Model not found: {m}\nCheck your dataset attachment and filename.')

def start_server(model_path, port, ctx, label):
    log_file = LOG_DIR / f'server_{port}.log'
    cmd = [
        SERVER_BIN,
        '-m', str(model_path),
        '--port', str(port),
        '--host', '127.0.0.1',
        '-c', str(ctx),
        '-ngl', '99',    # offload all layers to GPU
        '--log-disable', # suppress verbose model output to log file
    ]
    log_f = open(log_file, 'w')
    proc = subprocess.Popen(cmd, stdout=log_f, stderr=log_f)
    print(f'[{label}] Server PID {proc.pid} starting on port {port} → log: {log_file}')
    return proc

def wait_for_server(port, label, timeout=120):
    url = f'http://127.0.0.1:{port}/health'
    for i in range(timeout):
        try:
            urllib.request.urlopen(url, timeout=2)
            print(f'[{label}] Ready on port {port} (took {i}s)')
            return True
        except Exception:
            time.sleep(1)
    raise TimeoutError(f'{label} did not start within {timeout}s — check {LOG_DIR}/server_{port}.log')

# Start both servers
junior_proc = start_server(JUNIOR_MODEL, 8080, 4096, 'Junior')
senior_proc = start_server(SENIOR_MODEL, 8081, 8192, 'Senior')

# Poll until ready
wait_for_server(8080, 'Junior')
wait_for_server(8081, 'Senior')

print('\n[OK] Both servers online and ready.')

# Store PIDs so cleanup cell can stop them
os.environ['JUNIOR_PID'] = str(junior_proc.pid)
os.environ['SENIOR_PID'] = str(senior_proc.pid)

Cell 4 — Install Python dependencies


In [ ]:
import subprocess, sys

packages = [
    'crewai',
    'crewai-tools',
    'agentops',
    'fastapi',
    'httpx',
    'pytest',
    'pytest-asyncio',
]

print('Installing packages...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q'] + packages,
    check=True
)
print('[OK] All packages installed.')

Cell 5 — Copy MetroStack codebase to working directory


In [ ]:
import shutil
from pathlib import Path

SRC  = Path('/kaggle/input/metrostack')
DEST = Path('/kaggle/working/MetroStack')

# Kaggle input is read-only — copy to working dir so agents can write test files
if DEST.exists():
    shutil.rmtree(DEST)
shutil.copytree(SRC, DEST)

# Create audit test output directory
(DEST / 'tests' / 'audit').mkdir(parents=True, exist_ok=True)

print(f'[OK] Codebase copied to: {DEST}')
print(f'     Backend dir: {DEST / "backend"}')
print(f'     Writable: {os.access(DEST, os.W_OK)}')

# Expose path for the crew script
os.environ['METROSTACK_WORKING_DIR'] = str(DEST)

Cell 6 — Run the Audit Crew


In [ ]:
import os, sys, time, json, subprocess, threading
from datetime import datetime
from pathlib import Path
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
from crewai.agents.parser import AgentAction, AgentFinish
import agentops

# ── Kaggle paths (Linux) replacing Windows paths ──────────────────
METROSTACK_DIR = Path(os.environ.get('METROSTACK_WORKING_DIR', '/kaggle/working/MetroStack'))
BACKEND_DIR    = METROSTACK_DIR / 'backend'
TESTS_DIR      = METROSTACK_DIR / 'tests' / 'audit'
AUDIT_REPORT   = METROSTACK_DIR / 'audit_report.md'
SESSION_STATE  = METROSTACK_DIR / 'audit_session_state.json'
VENV_PYTHON    = sys.executable   # use the notebook's own Python — no venv needed on Kaggle
MAX_RETRIES    = 3

TESTS_DIR.mkdir(parents=True, exist_ok=True)

# ── AgentOps (reads from Kaggle secret if set) ────────────────────
AGENTOPS_KEY = os.environ.get('AGENTOPS_API_KEY', '214bffc9-2e26-4854-8118-ec5bc0e50391')
agentops.init(api_key=AGENTOPS_KEY, default_tags=['metro-stack-crew', 'kaggle-gpu', 'audit'])

# ── LLMs — same ports, now with GPU via -ngl 99 ───────────────────
senior_llm = LLM(
    model='openai/deepseek-coder-1.3b-instruct-q4_k_m',
    base_url='http://127.0.0.1:8081/v1',
    api_key='not-needed',
    temperature=0.1,
    max_tokens=4096,
    timeout=180,
)

junior_llm = LLM(
    model='openai/qwen2.5-coder-1.5b-instruct-q4_k_m',
    base_url='http://127.0.0.1:8080/v1',
    api_key='not-needed',
    temperature=0.2,
    max_tokens=4096,
    timeout=180,
)

# ── Heartbeat — shows you agents are alive ────────────────────────
_last_activity  = [time.time()]
_heartbeat_stop = threading.Event()

def _heartbeat_worker(interval=15):
    while not _heartbeat_stop.wait(interval):
        idle = int(time.time() - _last_activity[0])
        ts   = datetime.now().strftime('%H:%M:%S')
        print(f'  [{ts}] ... still running (last activity {idle}s ago)', flush=True)

def _touch():
    _last_activity[0] = time.time()

def make_step_callback(fid):
    step = [0]
    def on_step(out):
        _touch()
        step[0] += 1
        ts = datetime.now().strftime('%H:%M:%S')
        if isinstance(out, AgentAction):
            t = getattr(out, 'tool', '?')
            i = str(getattr(out, 'tool_input', ''))[:120].replace('\n', ' ')
            print(f'  [{ts}] [{fid}] Step {step[0]:02d} | TOOL -> {t}({i})', flush=True)
        elif isinstance(out, AgentFinish):
            o = str(getattr(out, 'output', ''))[:150].replace('\n', ' ')
            print(f'  [{ts}] [{fid}] Step {step[0]:02d} | DONE -> {o}', flush=True)
        else:
            print(f'  [{ts}] [{fid}] Step {step[0]:02d} | {type(out).__name__}', flush=True)
    return on_step

def make_task_callback(fid):
    def on_task(out):
        _touch()
        ts = datetime.now().strftime('%H:%M:%S')
        s  = str(out)[:200].replace('\n', ' ')
        print(f'\n  [{ts}] [{fid}] TASK COMPLETE -> {s}\n', flush=True)
    return on_task

# ── Tools ─────────────────────────────────────────────────────────
@tool('read_file')
def read_file(relative_path: str) -> str:
    """Read a file from the MetroStack backend directory.
    Pass path relative to backend root e.g. 'routers/projects.py'
    Returns file contents."""
    target = BACKEND_DIR / relative_path.lstrip('/\\\\')
    if not target.exists():
        return f'ERROR: File not found: {target}'
    try:
        return target.read_text(encoding='utf-8')
    except Exception as e:
        return f'ERROR reading {target}: {e}'

@tool('list_directory')
def list_directory(relative_path: str = '') -> str:
    """List all files in the MetroStack backend directory.
    Pass path relative to backend root, or empty string for root."""
    target = BACKEND_DIR / relative_path.lstrip('/\\\\')
    if not target.exists():
        return f'ERROR: Directory not found: {target}'
    lines = []
    for p in sorted(target.rglob('*')):
        if any(part in {'.venv','venv','__pycache__','.git','node_modules'}
               for part in p.parts):
            continue
        indent = '  ' * (len(p.relative_to(target).parts) - 1)
        lines.append(f"{indent}{p.name}{'/' if p.is_dir() else ''}")
    return '\n'.join(lines) if lines else '(empty)'

@tool('write_test_file')
def write_test_file(filename: str, content: str) -> str:
    """Write a pytest test file into the audit tests directory.
    filename: just the filename e.g. 'test_projects.py'"""
    dest = TESTS_DIR / filename
    try:
        dest.write_text(content, encoding='utf-8')
        return f'OK: Written to {dest}'
    except Exception as e:
        return f'ERROR: {e}'

@tool('run_pytest')
def run_pytest(filename: str) -> str:
    """Run a pytest file from the audit tests directory.
    filename: just the filename e.g. 'test_projects.py'"""
    test_file = TESTS_DIR / filename
    if not test_file.exists():
        return f'ERROR: Test file not found: {test_file}'
    try:
        result = subprocess.run(
            [VENV_PYTHON, '-m', 'pytest', str(test_file), '-v', '--tb=short', '--no-header'],
            capture_output=True, text=True,
            cwd=str(METROSTACK_DIR), timeout=120,
        )
        output = result.stdout + result.stderr
        return output if output.strip() else '(no output)'
    except subprocess.TimeoutExpired:
        return 'ERROR: pytest timed out after 120 seconds'
    except Exception as e:
        return f'ERROR: {e}'

# ── Feature definitions (same as local version) ───────────────────
AUDIT_FEATURES = [
    {'id':'PROJ-01','name':'Project Management','description':"""Verify by reading source and writing tests:\n  - Create project (POST) with part_number, revision, description\n  - List all projects (GET) returns correct schema\n  - Update project fields (PATCH/PUT)\n  - Delete a project (DELETE)\n  - Status cycles: pending -> ready -> aligned -> analyzed\nKey files: routers/projects.py, models.py, schemas.py"""},
    {'id':'FILE-01','name':'File Upload & Processing','description':"""Verify by reading source and writing tests:\n  - Upload accepts CAD: STL, OBJ, PLY, STEP, IGES\n  - Upload accepts Scans: E57, LAS, LAZ, PLY, PCD, XYZ, CSV\n  - File size limit 2 GB enforced\n  - Mesh repair: fill holes, fix normals, remove degenerates\n  - Background task created on upload\n  - Status polling endpoint exists\nKey files: routers/files.py or upload.py, services/mesh_repair.py, tasks.py"""},
    {'id':'ALIGN-01','name':'Alignment','description':"""Verify by reading source and writing tests:\n  - ICP alignment Point-to-Plane 200 iterations\n  - FGR alignment using FPFH features\n  - Datum-constrained locks DOF\n  - Returns RMSE, fitness score, inlier count\n  - 4x4 transformation matrix stored\nKey files: routers/alignment.py, services/alignment.py"""},
    {'id':'DEV-01','name':'Deviation Analysis','description':"""Verify by reading source and writing tests:\n  - Signed point-to-CAD distance (+ outside, - inside)\n  - Statistics: Min, Max, Mean, RMS, P95, P99\n  - Bulk PostgreSQL insert for large point sets\n  - Color scale bounds configurable\n  - Heatmap data endpoint exists\nKey files: routers/deviation.py, services/deviation.py"""},
    {'id':'GDT-01','name':'GD&T Analysis','description':"""Verify by reading source and writing tests:\n  - All 11: Flatness, Straightness, Circularity, Cylindricity, Position,\n    Parallelism, Perpendicularity, Angularity, Profile of Surface,\n    Circular Runout, Total Runout\n  - Minimum Zone fitting (not least-squares)\n  - Returns measured_value, tolerance, pass_fail\nKey files: routers/gdt.py, services/gdt.py"""},
    {'id':'WALL-01','name':'Wall Thickness','description':"""Verify by reading source and writing tests:\n  - Ray-cast thickness, default 20k samples, configurable\n  - Returns Min, Max, Mean, Std Dev\n  - Viridis colormap per point\n  - Open boundary detection\nKey files: routers/thickness.py, services/thickness.py"""},
    {'id':'VIZ-01','name':'3D Visualization','description':"""Verify by reading source and writing tests:\n  - Endpoints serve mesh/point-cloud for Three.js\n  - Deviation heatmap returns per-vertex colors\n  - Wall thickness heatmap returns per-point colors\n  - Point cloud filters to 500k max\n  - Mesh returns vertices + faces\nKey files: routers/visualization.py or mesh.py"""},
]

# ── Agents ────────────────────────────────────────────────────────
auditor = Agent(
    role='Source Code Auditor',
    goal='Use list_directory and read_file tools to audit source. Report FOUND/MISSING/PARTIAL per sub-feature with file:function evidence.',
    backstory='Senior code reviewer specializing in FastAPI backends. Methodical and evidence-based.',
    llm=senior_llm, tools=[read_file, list_directory], verbose=True, allow_delegation=False, max_iter=8,
)

tester = Agent(
    role='Test Engineer',
    goal='Write pytest file with write_test_file, run with run_pytest, fix import errors by reading source, report results.',
    backstory='QA engineer expert in offline FastAPI pytest suites. Mocks SQLAlchemy, uses TestClient.',
    llm=junior_llm, tools=[write_test_file, run_pytest, read_file], verbose=True, allow_delegation=False, max_iter=6,
)

qa_gate = Agent(
    role='QA Gate',
    goal='Read pytest output. Output: GATE: PASS\nCOVERAGE: <list> OR GATE: FAIL\nREASON: <sentence>',
    backstory='Strict release gate. All tests must pass, no import errors, at least one test run.',
    llm=senior_llm, verbose=True, allow_delegation=False, max_iter=2,
)

# ── Session state ─────────────────────────────────────────────────
def load_session_state():
    if SESSION_STATE.exists():
        try:
            return json.loads(SESSION_STATE.read_text(encoding='utf-8'))
        except Exception:
            pass
    return {}

def save_session_state(results):
    state = {r['id']: r for r in results}
    SESSION_STATE.write_text(json.dumps(state, indent=2, default=str), encoding='utf-8')

# ── Feature runner ────────────────────────────────────────────────
def run_audit(feature):
    fid, fname, fdesc = feature['id'], feature['name'], feature['description']
    result = {'id':fid,'name':fname,'status':'flagged','attempts':0,'reason':'','coverage':'','output':''}
    step_cb = make_step_callback(fid)
    task_cb = make_task_callback(fid)

    for attempt in range(1, MAX_RETRIES + 1):
        result['attempts'] = attempt
        print(f"\n{'='*60}\n  Auditing : {fid} - {fname}  |  Attempt {attempt}/{MAX_RETRIES}\n{'='*60}\n")

        audit_task = Task(
            description=f"Feature: {fid} - {fname}\nBackend: {BACKEND_DIR}\n\n1. Call list_directory('') to get file tree.\n2. Read every relevant file.\n3. State FOUND/MISSING/PARTIAL per sub-feature with file:function.\n\nRequirements:\n{fdesc}\n\nAttempt {attempt}/{MAX_RETRIES}.",
            expected_output='Line-by-line audit: FOUND/MISSING/PARTIAL with evidence.',
            agent=auditor,
        )
        test_fn = f"test_{fid.lower().replace('-','_')}.py"
        test_task = Task(
            description=f"Feature: {fid} - {fname}\n\n1. write_test_file('{test_fn}', <content>) — import from backend, mock SQLAlchemy, use TestClient, one test per FOUND sub-feature.\n2. run_pytest('{test_fn}') and report output.\n3. If imports fail, read_file to check paths, fix, rerun.\n\nAttempt {attempt}/{MAX_RETRIES}.",
            expected_output='Full pytest output with pass/fail counts.',
            agent=tester, context=[audit_task],
        )
        gate_task = Task(
            description=f"Feature: {fid}\n\nPASS criteria: all tests pass, no ImportError, no collection errors, at least 1 test ran.\n\nOutput exactly:\nGATE: PASS\nCOVERAGE: <confirmed sub-features>\nOR\nGATE: FAIL\nREASON: <one sentence>\n\nAttempt {attempt}/{MAX_RETRIES}.",
            expected_output='GATE: PASS with COVERAGE, or GATE: FAIL with REASON.',
            agent=qa_gate, context=[audit_task, test_task],
        )

        crew = Crew(
            agents=[auditor, tester, qa_gate],
            tasks=[audit_task, test_task, gate_task],
            process=Process.sequential,
            verbose=True,
            full_output=True,
            step_callback=step_cb,
            task_callback=task_cb,
        )

        try:
            output = str(crew.kickoff())
            result['output'] = output
            for line in output.splitlines():
                if line.startswith('COVERAGE:'): result['coverage'] = line.replace('COVERAGE:','').strip()
                if line.startswith('REASON:'):   result['reason']   = line.replace('REASON:','').strip()
            if 'GATE: PASS' in output:
                result['status'] = 'passed'
                print(f'\n  GATE PASSED - {fid} verified on attempt {attempt}\n')
                break
            else:
                print(f"\n  GATE FAILED (attempt {attempt}) - {result['reason']}\n")
                if attempt == MAX_RETRIES:
                    print(f'  Flagged after {MAX_RETRIES} attempts - moving on\n')
        except Exception as exc:
            result['output'] = result['reason'] = str(exc)
            print(f'\n  Crew error on attempt {attempt}: {exc}\n')
            if attempt == MAX_RETRIES:
                result['status'] = 'flagged'
        time.sleep(2)
    return result

def write_audit_report(results):
    passed  = [r for r in results if r['status'] == 'passed']
    flagged = [r for r in results if r['status'] == 'flagged']
    lines = [
        '# MetroStack Audit Report',
        f"**Date:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        f'**Backend:** `{BACKEND_DIR}`',
        f'**Environment:** Kaggle GPU',
        '',
        '## Summary',
        '| | Count |','|---|---|',
        f'| Features audited | {len(results)} |',
        f'| Verified | {len(passed)} |',
        f'| Flagged  | {len(flagged)} |','',
        '## Results','',
    ]
    for r in results:
        icon = 'PASS' if r['status'] == 'passed' else 'FLAGGED'
        lines += [f"### [{icon}] {r['id']} - {r['name']}",
                  f"- Attempts: {r['attempts']}/{MAX_RETRIES}"]
        if r['coverage']: lines.append(f"- Verified: {r['coverage']}")
        if r['reason']:   lines.append(f"- Reason: {r['reason']}")
        lines.append('')
    report = '\n'.join(lines)
    AUDIT_REPORT.write_text(report, encoding='utf-8')
    print(f'\n[+] Report written: {AUDIT_REPORT}\n')
    print(report)

# ── MAIN ──────────────────────────────────────────────────────────
print('\n[+] MetroStack Audit Pipeline — Kaggle GPU')
print(f'    Backend  : {BACKEND_DIR}')
print(f'    Tests    : {TESTS_DIR}')
print(f'    Features : {len(AUDIT_FEATURES)}')

previous = load_session_state()
if previous:
    done    = [fid for fid,r in previous.items() if r['status']=='passed']
    flagged = [fid for fid,r in previous.items() if r['status']=='flagged']
    print(f'\n[+] Resuming — Passed: {done or "none"} | Flagged (retry): {flagged or "none"}')
else:
    print('\n[+] No previous session — starting fresh.')

features_to_run = []
audit_results   = []
for feature in AUDIT_FEATURES:
    fid = feature['id']
    if fid in previous and previous[fid]['status'] == 'passed':
        print(f'    SKIP  {fid} — already passed')
        audit_results.append(previous[fid])
    else:
        status = 'retry (was flagged)' if fid in previous else 'not yet run'
        print(f'    QUEUE {fid} — {status}')
        features_to_run.append(feature)

print(f'\n[+] Running {len(features_to_run)} feature(s)...\n')

# Start heartbeat
_heartbeat_stop.clear()
hb = threading.Thread(target=_heartbeat_worker, args=(15,), daemon=True)
hb.start()

try:
    for feature in features_to_run:
        result = run_audit(feature)
        audit_results.append(result)
        save_session_state(audit_results)
        time.sleep(3)
    write_audit_report(audit_results)
    agentops.end_session('Success')
except KeyboardInterrupt:
    print('\n[!] Interrupted — progress saved.')
    save_session_state(audit_results)
    write_audit_report(audit_results)
    agentops.end_session('Fail')
except Exception as exc:
    print(f'\n[!] Crashed: {exc} — progress saved.')
    save_session_state(audit_results)
    write_audit_report(audit_results)
    agentops.end_session('Fail')
    raise
finally:
    _heartbeat_stop.set()

Cell 7 — Download results


In [ ]:
import shutil, os
from pathlib import Path

WORKING = Path('/kaggle/working/MetroStack')
OUT     = Path('/kaggle/working')

# Copy the key output files to /kaggle/working root
# Kaggle lets you download anything from /kaggle/working after the session

files_to_grab = [
    WORKING / 'audit_report.md',
    WORKING / 'audit_session_state.json',
]

# Also grab all written test files
test_files = list((WORKING / 'tests' / 'audit').glob('*.py'))

all_files = files_to_grab + test_files
print('Files available for download:')
for f in all_files:
    if f.exists():
        dest = OUT / f.name
        shutil.copy2(f, dest)
        size = dest.stat().st_size
        print(f'  {dest.name}  ({size:,} bytes)')
    else:
        print(f'  {f.name}  (NOT FOUND)')

print('\nTo download: Output tab (right panel) → click each file → Download')

Cell 8 — Stop servers (run when done)


In [ ]:
import os, signal

for label, env_var in [('Junior (8080)', 'JUNIOR_PID'), ('Senior (8081)', 'SENIOR_PID')]:
    pid = os.environ.get(env_var)
    if pid:
        try:
            os.kill(int(pid), signal.SIGTERM)
            print(f'[OK] Stopped {label} (PID {pid})')
        except ProcessLookupError:
            print(f'[--] {label} already stopped')
    else:
        print(f'[--] {label} PID not found in env')